In [ ]:
# Colab shell cell
!pip install -U sentence-transformers pdfplumber huggingface-hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 957.6 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 29.7 MB/s eta 0:00:00


In [ ]:
# Colab python cell: upload one or more files from your computer
from google.colab import files
uploaded = files.upload()  # choose the PDF from your machine
# Get the uploaded filename
fname = list(uploaded.keys())[0]
print("Uploaded file:", fname)

Saving Lung-Cancer-Detection-using-Supervised-Machine-Learning-Techniques.pdf to Lung-Cancer-Detection-using-Supervised-Machine-Learning-Techniques.pdf
Uploaded file: Lung-Cancer-Detection-using-Supervised-Machine-Learning-Techniques.pdf


In [ ]:
# Colab python cell
import pdfplumber, re

def extract_abstract_from_pdf(path):
    txt = ""
    with pdfplumber.open(path) as pdf:
        for p in pdf.pages:
            page_text = p.extract_text()
            if page_text:
                txt += page_text + "\n"
    # Try to locate ABSTRACT -> stop at INTRODUCTION or next numbered section
    m = re.search(r'(?s)ABSTRACT[:\s\-]*\n*(.*?)(?:\n\s*(?:1\s*INTRODUCTION|INTRODUCTION|1\. INTRODUCTION|\n\s*I\. INTRODUCTION))', txt, re.IGNORECASE)
    if m:
        return m.group(1).strip()
    # fallback: try "ABSTRACT" to next uppercase header
    m2 = re.search(r'(?s)ABSTRACT[:\s\-]*\n*(.*?)(?:\n[A-Z]{3,}\s*\n)', txt, re.IGNORECASE)
    if m2:
        return m2.group(1).strip()
    # last fallback: return first ~300 words of the PDF
    return " ".join(txt.split()[:300])

pdf_path = fname  # or "/content/drive/.....pdf"
abstract_text = extract_abstract_from_pdf(pdf_path)
print("---- Extracted Abstract ----\n")
print(abstract_text[:1000], "\n\n---(truncated)---")

---- Extracted Abstract ----

In recent times, Lung cancer is the most common cause of mortality in both men and women
around the world. Lung cancer is the second most well-known disease after heart disease.
Although lung cancer prevention is impossible, early detection of lung cancer can effectively
treat lung cancer at an early stage. The possibility of a patient's survival rate increasing if lung
cancer is identified early. To detect and diagnose lung cancer in its early stages, a variety of
data analysis and machine learning techniques have been applied. In this paper, we applied
supervised machine learning algorithms like SVM (Support vector machine), ANN (Artificial
neural networks), MLR (Multiple linear regression), and RF (random forest), to detect the early
stages of lung tumors. The main purpose of this study is to examine the success of machine
learning algorithms in detecting lung cancer at an early stage. When compared to all other
supervised machine learning algorithms, t

In [ ]:
# Colab python cell
from sentence_transformers import SentenceTransformer

model_name = "BAAI/bge-base-en"   # recommended target
fallback = "sentence-transformers/paraphrase-mpnet-base-v2"

try:
    model = SentenceTransformer(model_name)
    print("Loaded model:", model_name)
except Exception as e:
    print("Failed to load", model_name, "\nError:", e)
    print("Falling back to", fallback)
    model = SentenceTransformer(fallback)
    model_name = fallback
    print("Loaded model:", model_name)

# Move model to GPU if available
try:
    model.to('cuda')
    print("Model moved to GPU.")
except Exception:
    print("GPU not available or model not moved (running on CPU).")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded model: BAAI/bge-base-en
GPU not available or model not moved (running on CPU).


In [ ]:
# Colab python cell
# Encode the abstract and at least one question. Use convert_to_tensor=False to get numpy arrays.
abstract_emb = model.encode(abstract_text, convert_to_tensor=False, show_progress_bar=True)

# Example user question (you can change this)
user_question = "Which supervised machine learning algorithm performed best for early lung cancer detection in the study?"
question_emb = model.encode(user_question, convert_to_tensor=False)

print("Abstract embedding shape:", getattr(abstract_emb, "shape", None))
print("Question embedding shape:", getattr(question_emb, "shape", None))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Abstract embedding shape: (768,)
Question embedding shape: (768,)


In [ ]:
# Colab python cell
import numpy as np

def cosine_similarity(a, b):
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

score = cosine_similarity(abstract_emb, question_emb)
print(f"Cosine similarity (abstract vs question): {score:.4f}")

Cosine similarity (abstract vs question): 0.8944


In [ ]:
# Colab python cell
templates = [
    "Which ML algorithm had the highest accuracy for early-stage lung tumor detection?",
    "What supervised methods (SVM, ANN, MLR, RF) were applied and which was best?",
    "How well did Random Forest perform compared other to classifiers in detecting lung cancer?",
    "What is the reported accuracy of Random Forest in the paper?"
]

template_embs = model.encode(templates, convert_to_tensor=False, show_progress_bar=False)
for q, emb in zip(templates, template_embs):
    print(f"Q: {q}")
    print("  similarity:", round(cosine_similarity(abstract_emb, emb), 4))

Q: Which ML algorithm had the highest accuracy for early-stage lung tumor detection?
  similarity: 0.8673
Q: What supervised methods (SVM, ANN, MLR, RF) were applied and which was best?
  similarity: 0.8174
Q: How well did Random Forest perform compared other to classifiers in detecting lung cancer?
  similarity: 0.8649
Q: What is the reported accuracy of Random Forest in the paper?
  similarity: 0.8076


In [ ]:
# Colab python cell
templates = [
    "Which ML algorithm had the highest accuracy for early-stage lung cancer detection?",
    "What supervised methods (SVM, ANN, MLR, RF) were applied and which was best?",
    "What is Lung Cancer and what are its early symptomps?",
    "Machine learning techniques, including SVM, ANN, MLR, and Random Forest, are increasingly being applied in healthcare for early detection and diagnosis of lung cancer."
]

template_embs = model.encode(templates, convert_to_tensor=False, show_progress_bar=False)
for q, emb in zip(templates, template_embs):
    print(f"Q: {q}")
    print("  similarity:", round(cosine_similarity(abstract_emb, emb), 4))

Q: Which ML algorithm had the highest accuracy for early-stage lung cancer detection?
  similarity: 0.8781
Q: What supervised methods (SVM, ANN, MLR, RF) were applied and which was best?
  similarity: 0.8174
Q: What is Lung Cancer and what are its early symptomps?
  similarity: 0.8098
Q: Machine learning techniques, including SVM, ANN, MLR, and Random Forest, are increasingly being applied in healthcare for early detection and diagnosis of lung cancer.
  similarity: 0.9023


In [ ]:
# Colab python cell
templates = [
    "Which ML algorithm had the highest accuracy for lung cancer detection?",
    "What supervised methods were applied and which was best?",
    "What are its early symptomps?",
    "Machine learning techniques,SVM, ANN, MLR, and Random Forest,healthcare,lung cancer."
]

template_embs = model.encode(templates, convert_to_tensor=False, show_progress_bar=False)
for q, emb in zip(templates, template_embs):
    print(f"Q: {q}")
    print("  similarity:", round(cosine_similarity(abstract_emb, emb), 4))

Q: Which ML algorithm had the highest accuracy for lung cancer detection?
  similarity: 0.8585
Q: What supervised methods were applied and which was best?
  similarity: 0.7495
Q: What are its early symptomps?
  similarity: 0.7198
Q: Machine learning techniques,SVM, ANN, MLR, and Random Forest,healthcare,lung cancer.
  similarity: 0.878


In [ ]:
# Colab python cell
templates = [
    "What is the most common cause of mortality in men and women worldwide?",
    "Which disease ranks just after heart disease in prevalence?",
    "Is lung cancer prevention currently possible?",
    "How can early detection of lung cancer impact treatment outcomes?"
]

template_embs = model.encode(templates, convert_to_tensor=False, show_progress_bar=False)
for q, emb in zip(templates, template_embs):
    print(f"Q: {q}")
    print("  similarity:", round(cosine_similarity(abstract_emb, emb), 4))

Q: What is the most common cause of mortality in men and women worldwide?
  similarity: 0.77
Q: Which disease ranks just after heart disease in prevalence?
  similarity: 0.7654
Q: Is lung cancer prevention currently possible?
  similarity: 0.8102
Q: How can early detection of lung cancer impact treatment outcomes?
  similarity: 0.8593


In [ ]:
# Colab python cell: Full Hybrid RAG Pipeline Execution

import numpy as np
import pandas as pd

# --- 1. Entity and Intent Extraction (Simulation) ---
user_query = "What are the symptoms of Lung Cancer?"
print("--- 1. Entity and Intent Extraction (Graph Database Path) ---")
print(f"For the question: '{user_query}'")
print(f"Intent: Symptoms")
print(f"Entity: Lung Cancer")

# Derived entities from the document abstract (Vector DB Content)
print("\n--- 2. Derived Entities (Vector Database Content) ---")
print("Entities derived from the document abstract (Vector DB Content):")
print("* Disease/Condition: Lung cancer")
print("* Disease: Heart disease (referenced for ranking)")
print("* ML Algorithm: SVM (Support vector machine)")
print("* ML Algorithm: ANN (Artificial neural networks)")
print("* ML Algorithm: MLR (Multiple linear regression)")
print("* ML Algorithm: RF (Random forest)")

# --- 2. Scoring and Calculation ---

# Define Retrieval Scores (Raw Input from CSV)
score_data = {
    'Section': ['Introduction', 'Methodology', 'Algorithm and Design'],
    'Graph Score (G)': [100.0, 75.0, 50.0],
    'Vector Score (V)': [77.1, 78.0, 73.4]
}
df = pd.DataFrame(score_data)
print("\n--- 3. Raw Retrieval Scores ---")
print(df.to_markdown(index=False))

# Normalization and Fusion
min_g, max_g = df['Graph Score (G)'].min(), df['Graph Score (G)'].max()
df['Normalized Graph Score (G_norm)'] = (df['Graph Score (G)'] - min_g) / (max_g - min_g)

min_v, max_v = df['Vector Score (V)'].min(), df['Vector Score (V)'].max()
df['Normalized Vector Score (V_norm)'] = (df['Vector Score (V)'] - min_v) / (max_v - min_v)

df['Unified Score'] = df['Normalized Graph Score (G_norm)'] + df['Normalized Vector Score (V_norm)']
df_sorted = df.sort_values(by='Unified Score', ascending=False)

print("\n--- Unified Score and Section After Normalization ---")
print(df_sorted.to_markdown(index=False, floatfmt=".4f"))

top_n = 2
top_sections = df_sorted.head(top_n)['Section'].tolist()
print(f"\nTop {top_n} Sections Retrieved: {top_sections}")

# --- 3. Final Answer and Validation Simulation ---

# Context and Simulated Answer
section_a_content = "Lung Cancer symptoms include a persistent cough, shortness of breath, chest pain, and wheezing."
section_b_content = "Additional symptoms can include coughing up blood (hemoptysis), unexplained weight loss, and fatigue."
rag_context = f"Introduction: {section_a_content}\n Methodology: {section_b_content}"

final_answer_simulated = (
    "The symptoms of Lung Cancer include a persistent cough, shortness of breath, "
    "chest pain, and wheezing, plus additional symptoms like coughing up blood (hemoptysis), "
    "unexplained weight loss, and fatigue."
)

validation_output = "VALID. The generated answer includes all details from Section A and Section B and is fully grounded in the provided context."

print("\n--- 4. Final Generated Answer ---")
print(f"User Question: {user_query}")
print(f"Answer: {final_answer_simulated}")

print("\n--- 5. Validation Output ---")
print(validation_output)

--- 1. Entity and Intent Extraction (Graph Database Path) ---
For the question: 'What are the symptoms of Lung Cancer?'
Intent: Symptoms
Entity: Lung Cancer

--- 2. Derived Entities (Vector Database Content) ---
Entities derived from the document abstract (Vector DB Content):
* Disease/Condition: Lung cancer
* Disease: Heart disease (referenced for ranking)
* ML Algorithm: SVM (Support vector machine)
* ML Algorithm: ANN (Artificial neural networks)
* ML Algorithm: MLR (Multiple linear regression)
* ML Algorithm: RF (Random forest)

--- 3. Raw Retrieval Scores ---
| Section              |   Graph Score (G) |   Vector Score (V) |
|:---------------------|------------------:|-------------------:|
| Introduction         |               100 |               77.1 |
| Methodology          |                75 |               78   |
| Algorithm and Design |                50 |               73.4 |

--- Unified Score and Section After Normalization ---
| Section              |   Graph Score (G) |